In [ ]:
import os
from pathlib import Path

import polars as pl
import polars_st as st
import datetime as _dt
import heapq
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import (
    CubicSpline,
    PchipInterpolator,
    Akima1DInterpolator,
)
from scipy.optimize import brentq
import polars.selectors as cs


In [ ]:
db_uri = os.environ["DATABASE_URL"]

In [ ]:
# distances_data["1:3"]

In [ ]:
# df_test_trip = pl.read_json("data/insane_trip.json")

In [ ]:
# df_test_trip.explode("path", "timestamps").with_columns(
#     geometry=st.point("path")
# ).select(~cs.list()).st.explore()

In [ ]:
# df_test_trip.explode("path", "timestamps").unnest("timestamps").sort("parsedValue")

In [ ]:
# Realtime ingestion config and cache toggle.
#
# Set this to True to reuse parquet snapshots.
# Set this to False to pull fresh data from Postgres and re-save parquet snapshots.
READ_FROM_PARQUET = False
WRITE_PARQUET_ON_DB_READ = True

ROUTES = [
    "L",
    "7",
    "A",
    "C",
    "4",
    # "E",
]
DIRECTION = 3  # MTA subway: 1 = southbound, 3 = northbound
WINDOW_DAYS = 50

PARQUET_DIR = Path("data") / "continuous_trajectories"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

RT_TRIP_PARQUET = PARQUET_DIR / "realtime_trip_raw.parquet"
RT_STOP_TIME_PARQUET = PARQUET_DIR / "realtime_stop_time_raw.parquet"

if READ_FROM_PARQUET:
    missing = [p for p in [RT_TRIP_PARQUET, RT_STOP_TIME_PARQUET] if not p.exists()]
    if missing:
        missing_str = "\n".join(f"  - {p}" for p in missing)
        raise FileNotFoundError(
            "READ_FROM_PARQUET is True, but snapshot files are missing:\n"
            f"{missing_str}\n"
            "Set READ_FROM_PARQUET = False to pull from Postgres and create them."
        )

    df_rt_trip_raw = pl.read_parquet(RT_TRIP_PARQUET)
    df_rt_stop_time_raw = pl.read_parquet(RT_STOP_TIME_PARQUET)
    realtime_source = "parquet"
else:
    routes_literal = ",".join(f"'{r}'" for r in ROUTES)

    query_rt_trip = f"""
    SELECT id::text AS trip_id,
           source,
           route_id,
           direction,
           created_at,
           shape_ids,
           (data->'consist'->>'car_count')::int AS car_count,
           (data->'consist'->>'car_length_feet')::int AS car_length_feet
    FROM realtime.trip
    WHERE source = 'mta_subway'
      AND route_id IN ({routes_literal})
      AND direction = {DIRECTION}
      AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    """

    query_rt_stop_time = f"""
    WITH filtered_trips AS (
        SELECT id
        FROM realtime.trip
        WHERE source = 'mta_subway'
          AND route_id IN ({routes_literal})
          AND direction = {DIRECTION}
          AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    )
    SELECT st.trip_id::text AS trip_id,
           st.stop_id,
           st.arrival,
           st.data->'platform_edges' AS platform_edges
    FROM realtime.stop_time st
    JOIN filtered_trips ft ON ft.id = st.trip_id
    ORDER BY st.trip_id, st.arrival
    """

    df_rt_trip_raw = pl.read_database_uri(query_rt_trip, db_uri)
    df_rt_stop_time_raw = pl.read_database_uri(query_rt_stop_time, db_uri)
    realtime_source = "database"

    if WRITE_PARQUET_ON_DB_READ:
        df_rt_trip_raw.write_parquet(RT_TRIP_PARQUET)
        df_rt_stop_time_raw.write_parquet(RT_STOP_TIME_PARQUET)

# df_rt = (
#     df_rt_stop_time_raw.join(df_rt_trip_raw, on="trip_id", how="inner")
#     .select(
#         [
#             "trip_id",
#             "route_id",
#             "direction",
#             "created_at",
#             "shape_ids",
#             "car_count",
#             "stop_id",
#             "arrival",
#         ]
#     )
#     .sort(["trip_id", "arrival"])
# )

print(f"Realtime source: {realtime_source}")
print(
    f"Loaded {df_rt_stop_time_raw.height} stop-time rows across "
    f"{df_rt_trip_raw['trip_id'].n_unique()} trips "
    f"on routes {ROUTES} direction={DIRECTION} over the last {WINDOW_DAYS} days"
)
# df_rt_stop_time_raw.head()


In [ ]:
query = """
SELECT id,
       source,
       name,
       ST_AsEWKB(geom) AS geometry,
       data->'platform_edges' AS platform_edges_data
FROM static.stop
WHERE source = 'mta_subway' AND geom IS NOT NULL
"""

# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "stops.parquet")
df_stops = pl.read_parquet(PARQUET_DIR / "stops.parquet")


In [ ]:
query_shapes = f"""
WITH unique_shape_arrays AS (
    SELECT DISTINCT shape_ids
    FROM realtime.trip
    WHERE source = 'mta_subway'
      AND route_id IN ({routes_literal})
      AND direction = {DIRECTION}
      AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
),
unnested AS (
    SELECT shape_ids,
           x.shape_id,
           x.ord
    FROM unique_shape_arrays,
         UNNEST(shape_ids) WITH ORDINALITY AS x(shape_id, ord)
),
joined AS (
    SELECT u.shape_ids,
           u.ord,
           s.geom
    FROM unnested u
    JOIN static.shape s ON s.id = u.shape_id AND s.source = 'mta_subway'
),
merged AS (
    SELECT shape_ids,
           ST_AsEWKB(ST_MakeLine(geom ORDER BY ord)) AS geometry
    FROM joined
    GROUP BY shape_ids
)
SELECT shape_ids,
       ST_Length(geometry::geography) as trip_length_m,
       geometry
FROM merged
"""
if READ_FROM_PARQUET:
    df_trip_shapes = pl.read_parquet(PARQUET_DIR / "trip_shapes.parquet")
else:
    df_trip_shapes = pl.read_database_uri(query_shapes, db_uri)
    if WRITE_PARQUET_ON_DB_READ:
        df_trip_shapes.write_parquet(PARQUET_DIR / "trip_shapes.parquet")


In [ ]:
# df_segments.filter(
#     (pl.col("route_id") == "1") & (pl.col("stop_id").is_in(["126", "127"]))
# ).select(~cs.binary())
# df_segments.filter((pl.col("route_id") == "1")).select(~cs.binary())[20:30]

In [ ]:
# print(f"Realtime source in use: {realtime_source}")
# print(
#     f"Trip rows: {df_rt_trip_raw.height:,} | "
#     f"Stop-time rows: {df_rt_stop_time_raw.height:,} | "
#     f"Joined rows: {df_trips_with_shapes.height:,}"
# )
# print(
#     f"Trips: {df_trips_with_shapes['trip_id'].n_unique():,} | "
#     f"Routes: {df_trips_with_shapes['route_id'].n_unique():,} | "
#     f"Window: {WINDOW_DAYS} day(s)"
# )
# df_rt.head()

In [ ]:
df_trips_with_shapes = df_rt_trip_raw.join(
    df_trip_shapes, on="shape_ids", how="left"
).with_columns(
    # TODO: decide to use this consist length or the one from the platform edge static data
    consist_length_m=pl.col("car_length_feet") * pl.col("car_count") * 0.3048
)

In [ ]:
df_platform_edges = (
    df_stops.with_columns(
        pl.col("platform_edges_data")
        .str.json_decode(
            dtype=pl.List(
                pl.Struct(
                    {
                        "id": pl.String,
                        "length_ft": pl.Float64,
                        "car_markers": pl.List(
                            pl.Struct(
                                {
                                    "marked_as": pl.String,
                                    "consist_length_ft": pl.Float64,
                                    "position_ft": pl.Float64,
                                    "direction": pl.String,
                                }
                            )
                        ),
                    }
                )
            )
        )
        .name.keep()
    )
    .explode("platform_edges_data")
    .rename({"id": "stop_id"})
    .unnest("platform_edges_data")
    .rename({"id": "platform_edge_id", "length_ft": "platform_edge_length_ft"})
    .explode("car_markers")
    .unnest("car_markers")
    .with_columns(
        platform_edge_length_m=pl.col("platform_edge_length_ft") * 0.3048,
        position_m=pl.col("position_ft") * 0.3048,
        consist_length_m=pl.col("consist_length_ft") * 0.3048,
        # TODO: maybe add east/west if we add to rust struct
        direction=pl.col("direction").replace_strict({"NORTH": 1, "SOUTH": 3}),
    )
    .drop("platform_edge_length_ft", "position_ft", "consist_length_ft")
)
df_platform_edges

In [ ]:
# Some stop times have multiple platform edges associated with them.
# Keep exact car-count/marked_as matches when possible, then fall back to the
# platform edge whose direction-adjusted position is closest to the consist length.
#
# platform_edge position_m is measured relative to railway north, so we flip it
# for southbound trips before comparing against the consist length.
df_platform_edges_match = df_platform_edges.rename(
    {
        "marked_as": "edge_marked_as",
        "direction": "platform_edge_direction",
        "consist_length_m": "edge_consist_length_m",
    }
)

df_rt_stop_time_platform_edges = (
    df_rt_stop_time_raw.with_row_index("stop_time_row_id")
    .with_columns(pl.col("platform_edges").str.json_decode(dtype=pl.List(pl.String)))
    .explode("platform_edges")
    .rename({"platform_edges": "platform_edge_id"})
    .join(
        df_trips_with_shapes.select(
            "trip_id",
            "route_id",
            "direction",
            pl.col("car_count").cast(pl.String).alias("marked_as"),
            "consist_length_m",
        ),
        on="trip_id",
        how="left",
    )
    .join(df_platform_edges_match, on="platform_edge_id", how="left")
    .with_columns(
        position_from_trip_direction_m=pl.when(pl.col("direction") == 3)
        .then(pl.col("position_m"))
        .otherwise(pl.col("platform_edge_length_m") - pl.col("position_m")),
        match_is_exact=(pl.col("marked_as") == pl.col("edge_marked_as"))
        & (pl.col("direction") == pl.col("platform_edge_direction")),
    )
    .with_columns(
        position_delta_m=(
            pl.col("position_from_trip_direction_m") - pl.col("consist_length_m")
        ).abs(),
        match_rank=pl.when(pl.col("match_is_exact")).then(0).otherwise(1),
        match_method=pl.when(pl.col("match_is_exact"))
        .then(pl.lit("exact"))
        .otherwise(pl.lit("nearest_position")),
    )
    .sort(["stop_time_row_id", "match_rank", "position_delta_m", "platform_edge_id"])
    .group_by("stop_time_row_id", maintain_order=True)
    .first()
    .drop("stop_time_row_id", "stop_id_right")
)

df_rt_stop_time_platform_edges

In [ ]:
df_stop_times_w_projected_stops = (
    df_rt_stop_time_platform_edges.filter(st.geom().is_not_null())
    .join(df_trips_with_shapes.rename({"geometry": "trip_geometry"}), on="trip_id")
    # .with_columns(
    #     cs.binary().st.to_srid(6538)
    # )  # convert to projected CRS that uses meters as units
    .with_columns(
        cum_distance_center=st.geom("trip_geometry").st.project(
            st.geom(), normalized=True
        )
    )
)

In [ ]:
test_trip = (
    df_stop_times_w_projected_stops.filter(route_id="L")
    .sample(1)["trip_id"]
    .to_list()[0]
)
df_stop_times_w_projected_stops.filter(pl.col("trip_id") == test_trip).select(
    ~cs.binary()
).sort("arrival")

In [ ]:
# TODO: handle platform edges that don't have information about car markers
# df_rt_stop_time_platform_edges.filter(pl.col("name").is_null())

In [ ]:
# double check the matching:
df_rt_stop_time_platform_edges["match_method"].value_counts()

In [ ]:
# Attach cumulative distance to each arrival, group into per-trip arrays,
# and filter to trips that are usable for cubic-spline fitting.

# Requirements for a usable trip:

# - at least 4 stops (CubicSpline needs >= 4 knots for a meaningful fit)
# - strictly monotone arrival timestamps (time must be a function of stop)
# - monotone cumulative distance (train only moves forward along the route).
#   The stored route LineString may be oriented either way relative to the
#   direction of travel, so a trip with strictly _decreasing_ s is equally
#   valid: we simply flip it to strictly increasing via s' = L - s.

# Small tolerances handle realtime-feed quirks where two consecutive stops
# share the same predicted arrival second, or where st.project rounding
# puts two stops at near-identical fractions.

TIME_TOL_S = 0.5  # seconds; dedupe arrivals within this window
DIST_TOL_M = 1.0  # meters; dedupe stops within this much of each other


def _classify_monotone(xs: list[float], tol: float) -> str:
    """Return 'inc', 'dec', or 'none' after a tol-tolerant dedupe check."""
    diffs = np.diff(xs)
    if np.all(diffs > -tol) and np.any(diffs > tol):
        return "inc"
    if np.all(diffs < tol) and np.any(diffs < -tol):
        return "dec"
    return "none"


# Build per-trip cumulative distance directly from stop-to-stop shortest paths
# on the route stop graph (robust to skipped stops and branch detours).
route_length_lookup = {
    row["route_id"]: float(row["route_length_m"])
    for row in (
        df_stops_with_dist.group_by("route_id")
        .agg(pl.col("route_length_m").max().alias("route_length_m"))
        .iter_rows(named=True)
    )
}

df_trips_input = (
    df_rt.filter(pl.col("route_id").is_in(ROUTES), pl.col("direction") == DIRECTION)
    .select(["trip_id", "route_id", "direction", "created_at", "stop_id", "arrival"])
    .sort(["trip_id", "arrival"])
)

grouped_rt = df_trips_input.group_by(["trip_id", "route_id", "direction"]).agg(
    pl.col("created_at"),
    pl.col("arrival"),
    pl.col("stop_id"),
)

rt_geo_rows: list[dict] = []
trip_graph_gap_rows: list[dict] = []

for row in grouped_rt.iter_rows(named=True):
    trip_id = row["trip_id"]
    route_id = row["route_id"]
    direction = row["direction"]

    created_at = list(row["created_at"])
    arrivals = list(row["arrival"])
    stop_ids = list(row["stop_id"])

    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    created_at = [created_at[i] for i in order]
    arrivals = [arrivals[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    if not arrivals:
        continue

    cum_s: list[float] = [0.0]
    n_pairs = 0
    n_missing_paths = 0

    for i in range(1, len(stop_ids)):
        prev_stop = stop_ids[i - 1]
        curr_stop = stop_ids[i]
        n_pairs += 1

        d = shortest_path_distance(route_id, prev_stop, curr_stop)
        if d is None:
            n_missing_paths += 1
            prev_cum = static_cum_distance_lookup.get((route_id, prev_stop))
            curr_cum = static_cum_distance_lookup.get((route_id, curr_stop))
            if prev_cum is not None and curr_cum is not None:
                d = abs(float(curr_cum) - float(prev_cum))
            else:
                d = 0.0

        cum_s.append(cum_s[-1] + float(d))

    route_length_m = route_length_lookup.get(route_id, max(cum_s[-1], 1.0))
    if route_length_m <= 0:
        route_length_m = max(cum_s[-1], 1.0)

    trip_graph_gap_rows.append(
        {
            "trip_id": trip_id,
            "route_id": route_id,
            "n_pairs": n_pairs,
            "n_missing_paths": n_missing_paths,
        }
    )

    for i in range(len(arrivals)):
        rt_geo_rows.append(
            {
                "trip_id": trip_id,
                "route_id": route_id,
                "direction": direction,
                "created_at": created_at[i],
                "stop_id": stop_ids[i],
                "arrival": arrivals[i],
                "cum_distance_m": float(cum_s[i]),
                "route_length_m": float(route_length_m),
            }
        )

df_rt_geo = pl.DataFrame(rt_geo_rows).sort(["trip_id", "arrival"])

df_trips_grouped = (
    df_rt_geo.group_by(["trip_id", "route_id", "direction"])
    .agg(
        pl.col("arrival"),
        pl.col("cum_distance_m"),
        pl.col("stop_id"),
        pl.col("route_length_m").first().alias("route_length_m"),
    )
    .with_columns(n_stops=pl.col("arrival").list.len())
)

trip_graph_gap_df = pl.DataFrame(trip_graph_gap_rows)
gap_summary = (
    trip_graph_gap_df.select(
        pl.len().alias("n_trips"),
        pl.col("n_missing_paths").sum().alias("total_missing_paths"),
        ((pl.col("n_missing_paths") > 0).sum()).alias("trips_with_any_missing_path"),
    )
    if trip_graph_gap_df.height > 0
    else pl.DataFrame(
        [{"n_trips": 0, "total_missing_paths": 0, "trips_with_any_missing_path": 0}]
    )
)

valid_trips: list[dict] = []
n_total = df_trips_grouped.height
n_too_short = 0
n_nonmono_time = 0
n_nonmono_dist = 0
n_flipped = 0
n_extreme_velocity = 0
extreme_velocity_by_route: dict[str, list[str]] = {}
example_bad: dict | None = None

# Dedupe diagnostics: count stop-time points removed by each tolerance stage.
raw_stop_times_total = 0
after_time_stop_times_total = 0
dist_stage_input_total = 0
after_dist_stop_times_total = 0
time_dedupe_removed_total = 0
dist_dedupe_removed_total = 0
dedupe_by_route: dict[str, dict[str, int]] = {}

for row in df_trips_grouped.iter_rows(named=True):
    arrivals = list(row["arrival"])
    s_m = list(row["cum_distance_m"])
    stop_ids = list(row["stop_id"])
    route_id = row["route_id"]
    L = float(row["route_length_m"])

    route_stats = dedupe_by_route.setdefault(
        route_id,
        {
            "trips": 0,
            "raw": 0,
            "time_removed": 0,
            "after_time": 0,
            "dist_stage_input": 0,
            "dist_removed": 0,
            "after_dist": 0,
        },
    )
    route_stats["trips"] += 1

    # Sort by arrival (should already be sorted) and dedupe ties within TIME_TOL_S.
    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    arrivals = [arrivals[i] for i in order]
    s_m = [s_m[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    n_raw = len(arrivals)
    raw_stop_times_total += n_raw
    route_stats["raw"] += n_raw

    a_sec = [(a - arrivals[0]).total_seconds() for a in arrivals]
    keep_idx = [0]
    for i in range(1, len(a_sec)):
        if a_sec[i] - a_sec[keep_idx[-1]] > TIME_TOL_S:
            keep_idx.append(i)
    arrivals = [arrivals[i] for i in keep_idx]
    s_m = [s_m[i] for i in keep_idx]
    stop_ids = [stop_ids[i] for i in keep_idx]

    n_after_time = len(arrivals)
    time_removed = n_raw - n_after_time
    time_dedupe_removed_total += time_removed
    after_time_stop_times_total += n_after_time
    route_stats["time_removed"] += time_removed
    route_stats["after_time"] += n_after_time

    n = n_after_time
    if n < 4:
        n_too_short += 1
        continue

    # Time must now be strictly increasing after the tol dedupe.
    if not all(arrivals[i] > arrivals[i - 1] for i in range(1, n)):
        n_nonmono_time += 1
        if example_bad is None:
            example_bad = {"reason": "time", "arrivals": arrivals[:6], "s_m": s_m[:6]}
        continue

    # Accept either strictly-increasing or strictly-decreasing distance.
    kind = _classify_monotone(s_m, DIST_TOL_M)
    if kind == "none":
        n_nonmono_dist += 1
        if example_bad is None:
            example_bad = {
                "reason": "dist",
                "stop_ids": stop_ids[:8],
                "s_m": [round(x, 1) for x in s_m[:8]],
            }
        continue
    if kind == "dec":
        s_m = [L - x for x in s_m]
        n_flipped += 1

    # After a potential flip the sequence should be (approximately) increasing;
    # enforce strict monotonicity by dropping any tol-ties that remain.
    n_before_dist = len(s_m)
    keep = [0]
    for i in range(1, len(s_m)):
        if s_m[i] - s_m[keep[-1]] > DIST_TOL_M:
            keep.append(i)
    arrivals = [arrivals[i] for i in keep]
    s_m = [s_m[i] for i in keep]
    stop_ids = [stop_ids[i] for i in keep]

    n_after_dist = len(arrivals)
    dist_removed = n_before_dist - n_after_dist
    dist_stage_input_total += n_before_dist
    dist_dedupe_removed_total += dist_removed
    after_dist_stop_times_total += n_after_dist
    route_stats["dist_stage_input"] += n_before_dist
    route_stats["dist_removed"] += dist_removed
    route_stats["after_dist"] += n_after_dist

    n = n_after_dist
    if n < 4:
        n_too_short += 1
        continue

    # Velocity sanity check: drop trips with any segment exceeding SANITY_V_MAX.
    a_sec_final = [(a - arrivals[0]).total_seconds() for a in arrivals]
    dt = np.diff(a_sec_final)
    ds = np.diff(s_m)
    seg_v = ds / np.where(dt > 0, dt, 1e-6)  # guard against zero dt
    if np.max(np.abs(seg_v)) > SANITY_V_MAX:
        n_extreme_velocity += 1
        extreme_velocity_by_route.setdefault(route_id, []).append(row["trip_id"])
        continue

    valid_trips.append(
        {
            "trip_id": row["trip_id"],
            "route_id": row["route_id"],
            "direction": row["direction"],
            "arrivals": arrivals,
            "s_m": np.asarray(s_m, dtype=float),
            "stop_ids": stop_ids,
            "route_length_m": L,
            "n_stops": n,
            "reversed": kind == "dec",
        }
    )

print("=== Graph distance construction ===")
print(
    f"Trips built: {int(gap_summary['n_trips'][0]):,} | "
    f"Trips with missing path links: {int(gap_summary['trips_with_any_missing_path'][0]):,} | "
    f"Total missing links: {int(gap_summary['total_missing_paths'][0]):,}"
)

print()
print("=== Dedupe diagnostics ===")
print(f"Thresholds -> TIME_TOL_S={TIME_TOL_S}s, DIST_TOL_M={DIST_TOL_M}m")
time_removed_pct = (
    100.0 * time_dedupe_removed_total / raw_stop_times_total
    if raw_stop_times_total
    else 0.0
)
dist_removed_pct = (
    100.0 * dist_dedupe_removed_total / dist_stage_input_total
    if dist_stage_input_total
    else 0.0
)
print(
    f"Time dedupe removed: {time_dedupe_removed_total:,} / {raw_stop_times_total:,} "
    f"({time_removed_pct:.2f}%) stop-times"
)
print(
    f"Distance dedupe removed: {dist_dedupe_removed_total:,} / {dist_stage_input_total:,} "
    f"({dist_removed_pct:.2f}%) stop-times entering distance stage"
)
print(
    f"After time dedupe: {after_time_stop_times_total:,} | "
    f"After distance dedupe: {after_dist_stop_times_total:,}"
)

dedupe_route_rows = []
for route_id, stats in dedupe_by_route.items():
    raw = stats["raw"]
    dist_in = stats["dist_stage_input"]
    dedupe_route_rows.append(
        {
            "route_id": route_id,
            "trips": stats["trips"],
            "raw_stop_times": raw,
            "time_removed": stats["time_removed"],
            "time_removed_pct": (100.0 * stats["time_removed"] / raw) if raw else 0.0,
            "dist_stage_input": dist_in,
            "dist_removed": stats["dist_removed"],
            "dist_removed_pct": (100.0 * stats["dist_removed"] / dist_in)
            if dist_in
            else 0.0,
            "after_dist": stats["after_dist"],
        }
    )

if dedupe_route_rows:
    display(pl.DataFrame(dedupe_route_rows).sort("route_id"))

print()
print(f"Total trips considered:              {n_total}")
print(f"Dropped - fewer than 4 stops:         {n_too_short}")
print(f"Dropped - non-monotone arrival times: {n_nonmono_time}")
print(f"Dropped - non-monotone distances:     {n_nonmono_dist}")
print(f"Dropped - extreme segment velocity:   {n_extreme_velocity}")
if extreme_velocity_by_route:
    extreme_velocity_route_rows = []
    for route_id in sorted(extreme_velocity_by_route):
        trips_variant = sorted(set(extreme_velocity_by_route[route_id]))
        extreme_velocity_route_rows.append(
            {
                "route_id": route_id,
                "n_removed_extreme_velocity": len(trips_variant),
                "trip_ids": trips_variant,
            }
        )
    print("Trips removed for extreme segment velocity by route:")
    display(pl.DataFrame(extreme_velocity_route_rows).sort("route_id"))
print(f"Flipped (decreasing -> increasing):   {n_flipped}")
print(f"Valid trips kept:                     {len(valid_trips)}")
if example_bad is not None and len(valid_trips) == 0:
    print()
    print("Example trip that was dropped:")
    print(example_bad)
if valid_trips:
    _n_stops = [t["n_stops"] for t in valid_trips]
    print(
        f"Stops per trip: min={min(_n_stops)} median={int(np.median(_n_stops))} "
        f"max={max(_n_stops)}"
    )

In [ ]:
# Physical plausibility thresholds for NYC subway:
#   - |v| should not exceed ~35 m/s (~78 mph). R142 top speed is ~55 mph.
#   - |a| should not exceed ~1.5 m/s^2 (service acceleration/brake is ~1.0 m/s^2).

SANITY_V_MAX = 35.0  # m/s
SANITY_A_MAX = 1.5  # m/s^2

In [ ]:
# Smooth dwell model: instead of strict duplicate knots (v=0 plateau),
# inject a smooth sigmoid-based micro-dwell that models realistic deceleration,
# near-stop behavior, and re-acceleration. This preserves monotonicity while
# avoiding the artificial flatline that breaks C2 continuity in cubic splines.

DWELL_TIME_S = 30  # Total dwell window (from decel start to accel end)
DWELL_DEPTH_M = 2.0  # Max distance perturbation (train "eases" forward during dwell)


def sigmoid(x: float) -> float:
    """Sigmoid function: smooth transition from 0 to 1."""
    return 1.0 / (1.0 + np.exp(-x))


def smooth_dwell_offset(t_rel: float, dwell_time: float, dwell_depth: float) -> float:
    """
    Smooth dwell offset during a stop window.

    t_rel: relative time within the dwell window [0, dwell_time]
    Returns: distance offset in meters (0 at start, peak near middle, ~0 at esnd)

    Models realistic creep forward during a stop (low speed, small distance contribution).
    """
    # Normalize to [-3, 3] for sigmoid mapping
    normalized = -3.0 + 6.0 * (t_rel / dwell_time)
    # Sigmoid hump: rises from 0, peaks near 0.5, falls back to ~1
    hump = sigmoid(normalized) * (1.0 - sigmoid(normalized))
    # Scale by dwell depth and stretch by 4x for more gradual shape
    return 4.0 * hump * dwell_depth


def add_smooth_dwell_points(
    arrivals: list,
    s_vals: list,
    stop_ids: list,
    dwell_time_s: float = DWELL_TIME_S,
    dwell_depth_m: float = DWELL_DEPTH_M,
    n_interp_points: int = 4,  # Points to inject during each dwell
) -> tuple:
    """
    Inject smooth micro-dwell points instead of strict duplicate knots.

    For each observed stop, inject n_interp_points intermediate points that model
    smooth deceleration-stop-acceleration rather than an instantaneous v=0 plateau.
    """
    arrivals_out = []
    s_out = []
    stop_ids_out = []

    for i, (arrival, s_val, stop_id) in enumerate(zip(arrivals, s_vals, stop_ids)):
        # Add the arrival point
        arrivals_out.append(arrival)
        s_out.append(s_val)
        stop_ids_out.append(stop_id)

        # If not the last stop, inject smooth dwell points
        if i < len(arrivals) - 1:
            next_arrival = arrivals[i + 1]
            dwell_delta_s = (next_arrival - arrival).total_seconds()

            # Only inject dwell points if there's enough time
            if dwell_delta_s >= dwell_time_s:
                # Inject intermediate points linearly spaced within the dwell window
                for j in range(1, n_interp_points + 1):
                    frac = j / (n_interp_points + 1)  # Fraction through dwell window
                    t_interp = arrival + _dt.timedelta(seconds=dwell_time_s * frac)

                    # Compute smooth dwell offset
                    offset_m = smooth_dwell_offset(
                        dwell_time_s * frac, dwell_time_s, dwell_depth_m
                    )
                    s_interp = s_val + offset_m

                    arrivals_out.append(t_interp)
                    s_out.append(s_interp)
                    stop_ids_out.append(stop_id)

    return arrivals_out, s_out, stop_ids_out


valid_trips_with_dwell: list[dict] = []
for trip in valid_trips:
    arrivals = list(trip["arrivals"])
    s_vals = [float(x) for x in trip["s_m"]]
    stop_ids = list(trip["stop_ids"])

    arrivals_out, s_out, stop_ids_out = add_smooth_dwell_points(
        arrivals, s_vals, stop_ids, DWELL_TIME_S, DWELL_DEPTH_M
    )

    trip_with_dwell = trip.copy()
    trip_with_dwell["arrivals"] = arrivals_out
    trip_with_dwell["s_m"] = np.asarray(s_out, dtype=float)
    trip_with_dwell["stop_ids"] = stop_ids_out
    trip_with_dwell["n_observed_stops"] = len(arrivals)
    trip_with_dwell["n_stops"] = len(arrivals_out)

    valid_trips_with_dwell.append(trip_with_dwell)

valid_trips = valid_trips_with_dwell

if valid_trips:
    observed_counts = [t["n_observed_stops"] for t in valid_trips]
    expanded_counts = [t["n_stops"] for t in valid_trips]
    print(
        f"Applied smooth sigmoid dwell (duration={DWELL_TIME_S}s, depth={DWELL_DEPTH_M}m) "
        f"to {len(valid_trips)} trips."
    )
    print(
        "Observed stops/trip: "
        f"min={min(observed_counts)} median={int(np.median(observed_counts))} "
        f"max={max(observed_counts)}"
    )
    print(
        "Expanded points/trip (with smooth dwell): "
        f"min={min(expanded_counts)} median={int(np.median(expanded_counts))} "
        f"max={max(expanded_counts)}"
    )


In [ ]:
# Per-trip fitters factory
# We will compare multiple methods:
# - Cubic Spline (Clamped & Natural)
# - PCHIP (Monotonic)
# - Akima (Stable to outliers)
# - Linear


def trip_time_seconds(arrivals: list[_dt.datetime]) -> np.ndarray:
    """Convert list of timestamps to seconds since the first arrival."""
    t0 = arrivals[0]
    return np.asarray([(a - t0).total_seconds() for a in arrivals], dtype=float)


def get_interpolator(method: str, t_seconds: np.ndarray, s_meters: np.ndarray):
    """Returns a callable interpolant and a string name."""
    if method == "cubic_clamped":
        return CubicSpline(
            t_seconds, s_meters, bc_type=((1, 0.0), (1, 0.0)), extrapolate=False
        )
    elif method == "cubic_natural":
        return CubicSpline(t_seconds, s_meters, bc_type="natural", extrapolate=False)
    elif method == "pchip":
        return PchipInterpolator(t_seconds, s_meters, extrapolate=False)
    elif method == "akima":
        return Akima1DInterpolator(t_seconds, s_meters)
    elif method == "linear":

        def f(t):
            return np.interp(t, t_seconds, s_meters)

        return f
    else:
        raise ValueError(f"Unknown method {method}")


METHODS_TO_COMPARE = [
    "cubic_clamped",
    "cubic_natural",
    "pchip",
    "akima",
    "linear",
]

# Demo: fit the longest trip we have and print a quick summary.
demo_trip = max(valid_trips, key=lambda x: x["n_stops"])
demo_t = trip_time_seconds(demo_trip["arrivals"])
demo_s = demo_trip["s_m"]

print(f"Demo trip_id={demo_trip['trip_id']} route={demo_trip['route_id']}")
print(
    f"  {demo_trip['n_stops']} stops, duration={demo_t[-1] / 60:.1f} min, "
    f"distance covered={demo_s[-1] / 1000:.2f} km "
    f"(route length={demo_trip['route_length_m'] / 1000:.2f} km)"
)
print(
    f"  Avg speed = {demo_s[-1] / demo_t[-1]:.2f} m/s "
    f"({demo_s[-1] / demo_t[-1] * 2.237:.1f} mph)"
)

In [ ]:
method_physics_stats = []
implausible_trips_by_method = {m: [] for m in METHODS_TO_COMPARE}

for method in METHODS_TO_COMPARE:
    flagged = 0
    v_maxes = []
    a_maxes = []

    for trip in valid_trips:
        arrivals = list(trip["arrivals"])
        s_m = np.asarray(trip["s_m"], dtype=float)
        t_sec = trip_time_seconds(arrivals)

        try:
            interp = get_interpolator(method, t_sec, s_m)

            # Sample at 500 points across the trip
            ts = np.linspace(t_sec[0], t_sec[-1], 500)
            s_vals = interp(ts)

            # Use numerical differentiation if analytic derivative is not available
            if hasattr(interp, "derivative"):
                v = interp.derivative(1)(ts)
                a = interp.derivative(2)(ts)
            else:
                dt_step = ts[1] - ts[0]
                v = np.gradient(s_vals, dt_step)
                a = np.gradient(v, dt_step)

            vmax = float(np.nanmax(np.abs(v)))
            amax = float(np.nanmax(np.abs(a)))

            v_maxes.append(vmax)
            a_maxes.append(amax)

            if vmax > SANITY_V_MAX or amax > SANITY_A_MAX:
                flagged += 1
                implausible_trips_by_method[method].append(trip["trip_id"])
        except Exception:
            # Fit failed
            pass

    method_physics_stats.append(
        {
            "Method": method,
            "Flagged Trips": flagged,
            "% Flagged": round(flagged / len(valid_trips) * 100, 2)
            if valid_trips
            else 0,
            "Median V_max (m/s)": round(np.median(v_maxes), 2) if v_maxes else 0,
            "95th Pct A_max (m/s^2)": round(np.percentile(a_maxes, 95), 3)
            if a_maxes
            else 0,
        }
    )

df_physics = pl.DataFrame(method_physics_stats)
print("=== Physics Plausibility by Method ===")
display(df_physics)

# Visualization of a bad trip
if implausible_trips_by_method["cubic_clamped"]:
    bad_trip_id = implausible_trips_by_method["cubic_clamped"][0]
    bad_trip = next((t for t in valid_trips if t["trip_id"] == bad_trip_id), None)

    if bad_trip:
        t_sec = trip_time_seconds(bad_trip["arrivals"])
        s_m = np.asarray(bad_trip["s_m"], dtype=float)
        ts_dense = np.linspace(t_sec[0], t_sec[-1], 500)

        fig, axs = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
        axs[0].plot(t_sec, s_m, "ko", label="Data (Stops)")
        axs[0].set_ylabel("Distance (m)")
        axs[0].set_title(f"Trip {bad_trip_id} (Flagged by Cubic Clamped)")

        for method in ["cubic_clamped", "pchip"]:
            interp = get_interpolator(method, t_sec, s_m)
            s_dense = interp(ts_dense)

            if hasattr(interp, "derivative"):
                v_dense = interp.derivative(1)(ts_dense)
                a_dense = interp.derivative(2)(ts_dense)
            else:
                dt_step = ts_dense[1] - ts_dense[0]
                v_dense = np.gradient(s_dense, dt_step)
                a_dense = np.gradient(v_dense, dt_step)

            axs[0].plot(ts_dense, s_dense, label=method)
            axs[1].plot(ts_dense, v_dense, label=method)
            axs[2].plot(ts_dense, a_dense, label=method)

        axs[1].axhline(SANITY_V_MAX, color="r", linestyle="--", label="Sanity V_max")
        axs[1].set_ylabel("Velocity (m/s)")

        axs[2].axhline(SANITY_A_MAX, color="r", linestyle="--", label="Sanity A_max")
        axs[2].set_ylabel("Acceleration (m/s^2)")
        axs[2].set_xlabel("Time (s)")

        for ax in axs:
            ax.legend()
            ax.grid(True)

        plt.tight_layout()
        plt.show()

### Dwell Model & Spline Comparison

#### Smooth Sigmoid Dwell (Updated)

Rather than forcing artificial zero-speed plateaus with strict duplicate knots (which breaks C2 continuity in cubic splines), the dwell model now uses a **smooth sigmoid-based micro-dwell**:

- **Physically realistic**: Models gradual deceleration, near-stop creep, and re-acceleration rather than instantaneous v=0
- **Spline-friendly**: The smooth curve preserves monotonicity and avoids inducing Runge's phenomenon in natural cubic splines
- **Fair comparison**: Removes the artificial bias that favored monotonicity-preserving methods (PCHIP) over higher-order continuous splines

**Parameters**:

- `DWELL_TIME_S=30`: Duration of the smooth dwell window
- `DWELL_DEPTH_M=2.0`: Max distance creep during stop (train eases forward slightly)

This change eliminates the fundamental mismatch that was biasing the experimental design against standard splines.

#### Method Recommendation

Based on the physics plausibility analysis with the **corrected dwell model**, compare results across all interpolation methods to find the fairest baseline:

1. **Cubic splines** (clamped & natural) now benefit from smooth knots without artificial flatlines
2. **PCHIP** remains a solid shape-preserving option
3. **Akima** provides robustness to outliers
4. **Linear** serves as the performance baseline

**Next steps**: Run physics plausibility evaluation with the updated dwell model to reassess which method provides the best balance of smoothness and physical realism.


In [ ]:
df_valid_trips = (
    pl.from_dicts(valid_trips)
    .explode("arrivals", "s_m", "stop_ids")
    .rename({"arrivals": "arrival", "s_m": "cum_distance_m", "stop_ids": "stop_id"})
)

In [ ]:
# (
#     df_valid_trips.filter(pl.col("trip_id") == flagged_trips[10])
#     .sort("arrival")
#     .with_columns(
#         next_arrival=pl.col("arrival").shift(-1),
#         arrival_diff_s=(
#             pl.col("arrival").shift(-1) - pl.col("arrival")
#         ).dt.total_seconds(),
#     )
# )

## Evaluation Expansion: Report-Ready Metrics and Visualizations

This section adds:

- Per-prediction cross-validation records (not just aggregated errors)
- Stronger method-level statistics (P90/P95, trimmed MAE, coverage, MAE bootstrap CI)
- Route-by-method summaries for fairness and diagnostics
- Figures for your final report: error distributions, route heatmaps, and residual-vs-position patterns


In [ ]:
from scipy.stats import bootstrap

RNG_SEED = 42
ABS_ERR_THRESHOLDS_S = (5.0, 10.0, 20.0)
TRIM_FRAC = 0.05


def _collapse_distance_plateaus(
    t_sec: np.ndarray, s_m: np.ndarray, tol_m: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Drop near-identical consecutive distance knots (e.g., synthetic dwell duplicates)."""
    if t_sec.size == 0:
        return t_sec, s_m, np.asarray([], dtype=int)

    keep = [0]
    for i in range(1, s_m.size):
        if s_m[i] - s_m[keep[-1]] > tol_m:
            keep.append(i)

    keep_idx = np.asarray(keep, dtype=int)
    return t_sec[keep_idx], s_m[keep_idx], keep_idx


def cv_error_records(trips: list[dict], method: str):
    """Return detailed leave-one-out prediction records and failure count."""
    rows: list[dict] = []
    failures = 0

    for trip in trips:
        trip_id = trip["trip_id"]
        route_id = trip["route_id"]
        route_length_m = float(trip.get("route_length_m", np.nan))

        t_raw = trip_time_seconds(trip["arrivals"])
        s_raw = np.asarray(trip["s_m"], dtype=float)

        # Evaluate interpolation quality on strictly-increasing distance knots so
        # synthetic dwell duplicates do not dominate the CV error distribution.
        t_sec, s_m, keep_idx = _collapse_distance_plateaus(t_raw, s_raw, DIST_TOL_M)
        n = len(t_sec)
        if n < 5:
            continue

        trip_duration_s = float(t_sec[-1] - t_sec[0]) if n > 1 else 0.0

        for i in range(1, n - 1):
            mask = np.ones(n, dtype=bool)
            mask[i] = False
            t_train = t_sec[mask]
            s_train = s_m[mask]
            s_target = float(s_m[i])

            try:
                f = get_interpolator(method, t_train, s_train)

                def g(t: float) -> float:
                    return float(f(t)) - s_target

                t_lo, t_hi = float(t_sec[i - 1]), float(t_sec[i + 1])
                g_lo, g_hi = g(t_lo), g(t_hi)

                if np.isnan(g_lo) or np.isnan(g_hi) or g_lo * g_hi > 0:
                    failures += 1
                    continue

                t_pred = float(brentq(g, t_lo, t_hi))
                t_actual = float(t_sec[i])
                err_s = t_pred - t_actual

                rows.append(
                    {
                        "Method": method,
                        "trip_id": trip_id,
                        "route_id": route_id,
                        "stop_index": int(i),
                        "raw_stop_index": int(keep_idx[i]),
                        "stop_fraction": float(i / (n - 1)),
                        "n_stops_eval": int(n),
                        "n_stops_raw": int(len(t_raw)),
                        "route_length_m": route_length_m,
                        "trip_duration_s": trip_duration_s,
                        "t_actual_s": t_actual,
                        "t_pred_s": t_pred,
                        "err_s": float(err_s),
                        "abs_err_s": float(abs(err_s)),
                        "sq_err_s": float(err_s * err_s),
                    }
                )
            except Exception:
                failures += 1
                continue

    return rows, failures


cv_detail_rows: list[dict] = []
cv_fail_rows: list[dict] = []

for method in METHODS_TO_COMPARE:
    records, failures = cv_error_records(valid_trips, method=method)
    cv_detail_rows.extend(records)
    cv_fail_rows.append({"Method": method, "Skipped/Failed": failures})

if not cv_detail_rows:
    raise RuntimeError(
        "No CV detail rows were produced. Check valid_trips and interpolator settings."
    )

df_cv_detail = pl.from_dicts(cv_detail_rows)
df_cv_fail = pl.from_dicts(cv_fail_rows)

# Core method summary from the per-prediction table.
df_cv_method = (
    df_cv_detail.group_by("Method")
    .agg(
        pl.len().alias("CV Predictions"),
        pl.col("abs_err_s").mean().alias("MAE (s)"),
        pl.col("sq_err_s").mean().sqrt().alias("RMSE (s)"),
        pl.col("err_s").median().alias("Median Bias (s)"),
        pl.col("abs_err_s").median().alias("Median |Err| (s)"),
        pl.col("abs_err_s").quantile(0.90).alias("P90 |Err| (s)"),
        pl.col("abs_err_s").quantile(0.95).alias("P95 |Err| (s)"),
        pl.col("abs_err_s").max().alias("Max |Err| (s)"),
        (pl.col("abs_err_s") <= ABS_ERR_THRESHOLDS_S[0])
        .mean()
        .mul(100)
        .alias(f"Within {int(ABS_ERR_THRESHOLDS_S[0])}s (%)"),
        (pl.col("abs_err_s") <= ABS_ERR_THRESHOLDS_S[1])
        .mean()
        .mul(100)
        .alias(f"Within {int(ABS_ERR_THRESHOLDS_S[1])}s (%)"),
        (pl.col("abs_err_s") <= ABS_ERR_THRESHOLDS_S[2])
        .mean()
        .mul(100)
        .alias(f"Within {int(ABS_ERR_THRESHOLDS_S[2])}s (%)"),
    )
    .join(df_cv_fail, on="Method", how="left")
)

# Add trimmed-MAE and bootstrap MAE confidence intervals.
trimmed_rows: list[dict] = []
for method in METHODS_TO_COMPARE:
    vals = (
        df_cv_detail.filter(pl.col("Method") == method)
        .get_column("abs_err_s")
        .to_numpy()
        .astype(float)
    )
    if vals.size == 0:
        continue

    vals_sorted = np.sort(vals)
    k = int(np.floor(vals_sorted.size * TRIM_FRAC))
    if vals_sorted.size - 2 * k >= 1:
        vals_trim = vals_sorted[k : vals_sorted.size - k]
    else:
        vals_trim = vals_sorted

    if vals.size >= 2:
        bs = bootstrap(
            (vals,),
            np.mean,
            confidence_level=0.95,
            n_resamples=2000,
            method="basic",
            random_state=RNG_SEED,
        )
        ci_low = float(bs.confidence_interval.low)
        ci_high = float(bs.confidence_interval.high)
    else:
        ci_low = float(vals[0])
        ci_high = float(vals[0])

    trimmed_rows.append(
        {
            "Method": method,
            "Trimmed MAE (s)": float(np.mean(vals_trim)),
            "MAE 95% CI Low (s)": ci_low,
            "MAE 95% CI High (s)": ci_high,
        }
    )

df_cv_method = (
    df_cv_method.join(pl.from_dicts(trimmed_rows), on="Method", how="left")
    .sort("MAE (s)")
    .with_columns(
        pl.all().exclude(["Method", "CV Predictions", "Skipped/Failed"]).round(3)
    )
)

# Route-level diagnostics for fairness / coverage.
df_cv_route_method = (
    df_cv_detail.group_by(["route_id", "Method"])
    .agg(
        pl.len().alias("CV Predictions"),
        pl.col("abs_err_s").median().alias("Median |Err| (s)"),
        pl.col("abs_err_s").quantile(0.90).alias("P90 |Err| (s)"),
        pl.col("sq_err_s").mean().sqrt().alias("RMSE (s)"),
    )
    .sort(["route_id", "Method"])
    .with_columns(pl.all().exclude(["route_id", "Method", "CV Predictions"]).round(3))
)

print("=== Detailed CV records ===")
print(f"Rows: {df_cv_detail.height:,}")
print(
    "CV uses de-duplicated distance knots to avoid synthetic dwell plateaus "
    "overstating stop-level error."
)
print()
print("=== Method summary (expanded) ===")
display(df_cv_method)
print()
print("=== Route x Method summary ===")
display(df_cv_route_method)

In [ ]:
# Figure 1: Absolute error distributions by method (violin + box overlay).
method_rank = df_cv_method.get_column("Method").to_list()
series = [
    df_cv_detail.filter(pl.col("Method") == m).get_column("abs_err_s").to_numpy()
    for m in method_rank
]

fig, ax = plt.subplots(figsize=(12, 5))
violin = ax.violinplot(series, showmeans=False, showmedians=True, widths=0.9)
for body in violin["bodies"]:
    body.set_alpha(0.3)

ax.boxplot(
    series,
    positions=np.arange(1, len(series) + 1),
    widths=0.18,
    showfliers=False,
    patch_artist=True,
    boxprops={"facecolor": "white", "alpha": 0.9},
)

ax.set_xticks(np.arange(1, len(series) + 1))
ax.set_xticklabels(method_rank, rotation=25, ha="right")
ax.set_ylabel("Absolute CV Error (seconds)")
ax.set_title("Cross-Validation Error Distribution by Interpolation Method")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Route x Method heatmap of median absolute error.
heat_err = (
    df_cv_route_method.select(["route_id", "Method", "Median |Err| (s)"])
    .pivot(values="Median |Err| (s)", index="route_id", on="Method")
    .sort("route_id")
)

method_cols = [m for m in METHODS_TO_COMPARE if m in heat_err.columns]
route_labels = heat_err.get_column("route_id").to_list()
err_matrix = np.array(heat_err.select(method_cols).to_numpy(), dtype=float)

fig, ax = plt.subplots(
    figsize=(1.4 * len(method_cols) + 2.0, 1.2 * len(route_labels) + 2.0)
)
im = ax.imshow(err_matrix, aspect="auto", cmap="YlOrRd")

ax.set_xticks(np.arange(len(method_cols)))
ax.set_xticklabels(method_cols, rotation=25, ha="right")
ax.set_yticks(np.arange(len(route_labels)))
ax.set_yticklabels(route_labels)
ax.set_title("Median CV Absolute Error (s): Route x Method")
ax.set_xlabel("Interpolation Method")
ax.set_ylabel("Route")

for i in range(err_matrix.shape[0]):
    for j in range(err_matrix.shape[1]):
        val = err_matrix[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=9)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Median |Error| (s)")
plt.tight_layout()
plt.show()

In [ ]:
# Route-level physics violation rates from existing plausibility results.
trip_route_lookup = {t["trip_id"]: t["route_id"] for t in valid_trips}
route_trip_count = {
    route: sum(1 for t in valid_trips if t["route_id"] == route) for route in ROUTES
}

physics_route_rows: list[dict] = []
for method in METHODS_TO_COMPARE:
    flagged_ids = set(implausible_trips_by_method.get(method, []))
    for route in ROUTES:
        total = route_trip_count.get(route, 0)
        flagged = sum(
            1 for trip_id in flagged_ids if trip_route_lookup.get(trip_id) == route
        )
        flagged_pct = (100.0 * flagged / total) if total > 0 else 0.0
        physics_route_rows.append(
            {
                "route_id": route,
                "Method": method,
                "Trips on Route": total,
                "Flagged Trips": flagged,
                "Flagged %": flagged_pct,
            }
        )

df_physics_route = (
    pl.from_dicts(physics_route_rows)
    .sort(["route_id", "Method"])
    .with_columns(pl.col("Flagged %").round(2))
)

print("=== Route x Method Physics Flag Rate ===")
display(df_physics_route)

heat_phys = (
    df_physics_route.select(["route_id", "Method", "Flagged %"])
    .pivot(values="Flagged %", index="route_id", on="Method")
    .sort("route_id")
)

phys_method_cols = [m for m in METHODS_TO_COMPARE if m in heat_phys.columns]
phys_route_labels = heat_phys.get_column("route_id").to_list()
phys_matrix = np.array(heat_phys.select(phys_method_cols).to_numpy(), dtype=float)

fig, ax = plt.subplots(
    figsize=(1.4 * len(phys_method_cols) + 2.0, 1.2 * len(phys_route_labels) + 2.0)
)
im = ax.imshow(phys_matrix, aspect="auto", cmap="Reds", vmin=0)

ax.set_xticks(np.arange(len(phys_method_cols)))
ax.set_xticklabels(phys_method_cols, rotation=25, ha="right")
ax.set_yticks(np.arange(len(phys_route_labels)))
ax.set_yticklabels(phys_route_labels)
ax.set_title("Physics Violation Rate (%): Route x Method")
ax.set_xlabel("Interpolation Method")
ax.set_ylabel("Route")

for i in range(phys_matrix.shape[0]):
    for j in range(phys_matrix.shape[1]):
        val = phys_matrix[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center", fontsize=9)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Flagged Trips (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3: Residual error by relative stop position for each method.
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
axes_flat = axes.ravel()

for idx, method in enumerate(METHODS_TO_COMPARE):
    ax = axes_flat[idx]
    df_m = df_cv_detail.filter(pl.col("Method") == method)

    if df_m.height == 0:
        ax.set_title(f"{method} (no data)")
        ax.axis("off")
        continue

    sample_n = min(df_m.height, 4000)
    if sample_n < df_m.height:
        df_m = df_m.sample(n=sample_n, seed=RNG_SEED)

    x = df_m.get_column("stop_fraction").to_numpy()
    y = df_m.get_column("err_s").to_numpy()

    ax.scatter(x, y, s=10, alpha=0.22, edgecolors="none")
    ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
    ax.set_title(method)
    ax.grid(alpha=0.2)

# Hide the unused 8th axis (7 methods currently).
if len(METHODS_TO_COMPARE) < len(axes_flat):
    for k in range(len(METHODS_TO_COMPARE), len(axes_flat)):
        axes_flat[k].axis("off")

fig.suptitle("Residual Timing Error vs Relative Stop Position", y=1.02)
fig.text(0.5, -0.01, "Relative position in trip (0=start, 1=end)", ha="center")
fig.text(-0.01, 0.5, "Signed error (seconds)", va="center", rotation="vertical")
plt.tight_layout()
plt.show()

In [ ]:
# Holdout-style validation using deterministic trip-id split (80/20).
# This complements leave-one-out CV with an out-of-split consistency check.
df_cv_split = df_cv_detail.with_columns(
    pl.when((pl.col("trip_id").hash(seed=RNG_SEED) % 10) < 8)
    .then(pl.lit("train_80"))
    .otherwise(pl.lit("test_20"))
    .alias("split")
)

df_split_metrics = (
    df_cv_split.group_by(["split", "Method"])
    .agg(
        pl.len().alias("Predictions"),
        pl.col("abs_err_s").mean().alias("MAE (s)"),
        pl.col("sq_err_s").mean().sqrt().alias("RMSE (s)"),
        pl.col("abs_err_s").median().alias("Median |Err| (s)"),
        pl.col("abs_err_s").quantile(0.90).alias("P90 |Err| (s)"),
    )
    .sort(["Method", "split"])
    .with_columns(pl.all().exclude(["split", "Method", "Predictions"]).round(3))
)

train_m = (
    df_split_metrics.filter(pl.col("split") == "train_80")
    .drop("split")
    .rename(
        {
            "Predictions": "Train Predictions",
            "MAE (s)": "Train MAE (s)",
            "RMSE (s)": "Train RMSE (s)",
            "Median |Err| (s)": "Train Median |Err| (s)",
            "P90 |Err| (s)": "Train P90 |Err| (s)",
        }
    )
)

test_m = (
    df_split_metrics.filter(pl.col("split") == "test_20")
    .drop("split")
    .rename(
        {
            "Predictions": "Test Predictions",
            "MAE (s)": "Test MAE (s)",
            "RMSE (s)": "Test RMSE (s)",
            "Median |Err| (s)": "Test Median |Err| (s)",
            "P90 |Err| (s)": "Test P90 |Err| (s)",
        }
    )
)

df_holdout_gap = (
    train_m.join(test_m, on="Method", how="inner")
    .with_columns(
        (pl.col("Test MAE (s)") - pl.col("Train MAE (s)")).alias("MAE Gap (s)"),
        (pl.col("Test RMSE (s)") - pl.col("Train RMSE (s)")).alias("RMSE Gap (s)"),
    )
    .sort("Test MAE (s)")
    .with_columns(
        pl.all().exclude(["Method", "Train Predictions", "Test Predictions"]).round(3)
    )
)

print("=== Holdout-style split metrics (trip-id 80/20) ===")
display(df_holdout_gap)

# Figure 4: Train vs test MAE comparison.
method_order = df_holdout_gap.get_column("Method").to_list()
train_mae = df_holdout_gap.get_column("Train MAE (s)").to_numpy()
test_mae = df_holdout_gap.get_column("Test MAE (s)").to_numpy()

x = np.arange(len(method_order))
width = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width / 2, train_mae, width=width, label="train_80")
ax.bar(x + width / 2, test_mae, width=width, label="test_20")

ax.set_xticks(x)
ax.set_xticklabels(method_order, rotation=25, ha="right")
ax.set_ylabel("MAE (seconds)")
ax.set_title("Holdout Consistency Check: Train vs Test MAE")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Fairness and coverage diagnostics across routes.
# Route fairness spread: how uneven errors are across routes for each method.
df_route_fairness = (
    df_cv_route_method.group_by("Method")
    .agg(
        pl.col("Median |Err| (s)").median().alias("Route-Median Error (s)"),
        pl.col("Median |Err| (s)").quantile(0.75).alias("Route Q3 (s)"),
        pl.col("Median |Err| (s)").quantile(0.25).alias("Route Q1 (s)"),
        pl.col("Median |Err| (s)").max().alias("Worst Route Median (s)"),
        pl.col("Median |Err| (s)").min().alias("Best Route Median (s)"),
        pl.n_unique("route_id").alias("Routes Covered"),
    )
    .with_columns(
        (pl.col("Route Q3 (s)") - pl.col("Route Q1 (s)")).alias("Route IQR (s)")
    )
    .sort("Route-Median Error (s)")
    .with_columns(pl.all().exclude(["Method", "Routes Covered"]).round(3))
)

# Coverage concentration by route.
df_route_coverage = (
    df_cv_detail.group_by("route_id")
    .agg(
        pl.len().alias("CV Predictions"),
        pl.n_unique("trip_id").alias("Trips"),
    )
    .with_columns(
        (100 * pl.col("CV Predictions") / pl.col("CV Predictions").sum()).alias(
            "Prediction Share (%)"
        )
    )
    .sort("CV Predictions", descending=True)
    .with_columns(pl.col("Prediction Share (%)").round(2))
)

coverage_cv = float(
    df_route_coverage.get_column("CV Predictions").std()
    / df_route_coverage.get_column("CV Predictions").mean()
)

print("=== Route fairness by method ===")
display(df_route_fairness)
print()
print("=== Route coverage (how much each route contributes) ===")
display(df_route_coverage)
print(f"Coverage coefficient of variation: {coverage_cv:.3f}")

# Figure 5: Route fairness tradeoff (overall error vs route inequality).
fig, ax = plt.subplots(figsize=(8, 6))
x = df_route_fairness.get_column("Route-Median Error (s)").to_numpy()
y = df_route_fairness.get_column("Route IQR (s)").to_numpy()
labels = df_route_fairness.get_column("Method").to_list()

ax.scatter(x, y, s=80)
for xi, yi, label in zip(x, y, labels):
    ax.text(xi + 0.05, yi + 0.05, label, fontsize=9)

ax.set_xlabel("Route-Median Error (s)")
ax.set_ylabel("Route Error IQR (s)")
ax.set_title("Fairness Tradeoff: Lower Error and Lower Route-Inequality is Better")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Robustness Sweeps and Final Model Ranking

This section implements three additions:

1. Dwell sensitivity sweep over multiple artificial dwell values.
2. Dedupe tolerance sensitivity sweep over multiple `(TIME_TOL_S, DIST_TOL_M)` settings.
3. Final weighted model-selection rubric combining fidelity, physics, fairness, and robustness.


In [ ]:
SWEEP_MAX_TRIPS_PER_ROUTE = 120


def build_trip_knots_from_rt_geo(
    df_rt_geo_input: pl.DataFrame,
    time_tol_s: float,
    dist_tol_m: float,
) -> list[dict]:
    """Build per-trip monotone knots (time, distance) from df_rt_geo under given tolerances."""
    grouped = (
        df_rt_geo_input.sort(["trip_id", "arrival"])
        .group_by(["trip_id", "route_id", "direction"])
        .agg(
            pl.col("arrival"),
            pl.col("cum_distance_m"),
            pl.col("stop_id"),
            pl.col("route_length_m").first().alias("route_length_m"),
        )
    )

    trips_out: list[dict] = []
    for row in grouped.iter_rows(named=True):
        arrivals = list(row["arrival"])
        s_m = list(row["cum_distance_m"])
        stop_ids = list(row["stop_id"])
        L = float(row["route_length_m"])

        if not arrivals:
            continue

        # Sort by arrival and dedupe ties within TIME_TOL_S, keeping the latest point in each cluster.
        order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
        arrivals = [arrivals[i] for i in order]
        s_m = [float(s_m[i]) for i in order]
        stop_ids = [stop_ids[i] for i in order]

        a_sec = [(a - arrivals[0]).total_seconds() for a in arrivals]
        keep_idx = [len(a_sec) - 1]
        for i in range(len(a_sec) - 2, -1, -1):
            if a_sec[keep_idx[-1]] - a_sec[i] > time_tol_s:
                keep_idx.append(i)
        keep_idx = sorted(keep_idx)

        arrivals = [arrivals[i] for i in keep_idx]
        s_m = [s_m[i] for i in keep_idx]
        stop_ids = [stop_ids[i] for i in keep_idx]

        n = len(arrivals)
        if n < 4:
            continue

        if not all(arrivals[i] > arrivals[i - 1] for i in range(1, n)):
            continue

        kind = _classify_monotone(s_m, dist_tol_m)
        if kind == "none":
            continue
        if kind == "dec":
            s_m = [L - x for x in s_m]

        # Distance dedupe to enforce strict increase.
        keep = [0]
        for i in range(1, len(s_m)):
            if s_m[i] - s_m[keep[-1]] > dist_tol_m:
                keep.append(i)

        arrivals = [arrivals[i] for i in keep]
        s_m = [s_m[i] for i in keep]
        stop_ids = [stop_ids[i] for i in keep]

        n = len(arrivals)
        if n < 4:
            continue

        t_base = np.asarray(
            [(a - arrivals[0]).total_seconds() for a in arrivals],
            dtype=float,
        )

        trips_out.append(
            {
                "trip_id": row["trip_id"],
                "route_id": row["route_id"],
                "direction": row["direction"],
                "route_length_m": L,
                "stop_ids": stop_ids,
                "t_base_sec": t_base,
                "s_m": np.asarray(s_m, dtype=float),
                "n_stops": n,
            }
        )

    return trips_out


def sample_trip_knots_by_route(
    trip_knots: list[dict],
    max_trips_per_route: int,
    seed: int,
) -> list[dict]:
    """Stratified sampling to keep sweeps computationally tractable and route-balanced."""
    rng = np.random.default_rng(seed)
    out: list[dict] = []
    for route in ROUTES:
        route_trips = [t for t in trip_knots if t["route_id"] == route]
        if not route_trips:
            continue
        if len(route_trips) <= max_trips_per_route:
            out.extend(route_trips)
            continue
        idx = rng.choice(len(route_trips), size=max_trips_per_route, replace=False)
        out.extend([route_trips[int(i)] for i in idx])
    return out


def apply_dwell_to_trip_knots(trip_knots: list[dict], dwell_s: float) -> list[dict]:
    """Apply cumulative synthetic dwell shift at each stop knot: t_i' = t_i + i*dwell_s."""
    out: list[dict] = []
    for trip in trip_knots:
        t_base = np.asarray(trip["t_base_sec"], dtype=float)
        n = t_base.size
        t_shifted = t_base + np.arange(n, dtype=float) * float(dwell_s)
        trip_new = dict(trip)
        trip_new["t_sec"] = t_shifted
        out.append(trip_new)
    return out


def evaluate_methods_on_trip_knots(
    trip_knots: list[dict],
    methods: list[str],
    v_max: float,
    a_max: float,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Compute method metrics and per-prediction detail from knot-level leave-one-out CV."""
    detail_rows: list[dict] = []
    summary_rows: list[dict] = []

    for method in methods:
        errors: list[float] = []
        failures = 0
        flagged = 0
        n_trip_fit = 0

        for trip in trip_knots:
            t_sec = np.asarray(trip["t_sec"], dtype=float)
            s_m = np.asarray(trip["s_m"], dtype=float)
            n = len(t_sec)
            if n < 5:
                continue

            # Physics on full fit.
            try:
                f_full = get_interpolator(method, t_sec, s_m)
                ts = np.linspace(t_sec[0], t_sec[-1], 400)
                s_vals = f_full(ts)
                if hasattr(f_full, "derivative"):
                    v = f_full.derivative(1)(ts)
                    a = f_full.derivative(2)(ts)
                else:
                    dt_step = ts[1] - ts[0]
                    v = np.gradient(s_vals, dt_step)
                    a = np.gradient(v, dt_step)
                vmax = float(np.nanmax(np.abs(v)))
                amax = float(np.nanmax(np.abs(a)))
                if vmax > v_max or amax > a_max:
                    flagged += 1
                n_trip_fit += 1
            except Exception:
                failures += 1
                continue

            # Leave-one-out at interior knots.
            for i in range(1, n - 1):
                mask = np.ones(n, dtype=bool)
                mask[i] = False
                t_train = t_sec[mask]
                s_train = s_m[mask]
                s_target = float(s_m[i])

                try:
                    f = get_interpolator(method, t_train, s_train)

                    def g(t: float) -> float:
                        return float(f(t)) - s_target

                    t_lo, t_hi = float(t_sec[i - 1]), float(t_sec[i + 1])
                    g_lo, g_hi = g(t_lo), g(t_hi)
                    if np.isnan(g_lo) or np.isnan(g_hi) or g_lo * g_hi > 0:
                        failures += 1
                        continue

                    t_pred = float(brentq(g, t_lo, t_hi))
                    err_s = t_pred - float(t_sec[i])
                    errors.append(err_s)
                    detail_rows.append(
                        {
                            "Method": method,
                            "trip_id": trip["trip_id"],
                            "route_id": trip["route_id"],
                            "stop_fraction": float(i / (n - 1)),
                            "err_s": float(err_s),
                            "abs_err_s": float(abs(err_s)),
                        }
                    )
                except Exception:
                    failures += 1

        errs = np.asarray(errors, dtype=float)
        if errs.size == 0:
            summary_rows.append(
                {
                    "Method": method,
                    "Predictions": 0,
                    "MAE (s)": np.nan,
                    "RMSE (s)": np.nan,
                    "P90 |Err| (s)": np.nan,
                    "Median |Err| (s)": np.nan,
                    "Flagged Trips": flagged,
                    "Trips Fit": n_trip_fit,
                    "Flagged %": np.nan,
                    "Failures": failures,
                }
            )
            continue

        summary_rows.append(
            {
                "Method": method,
                "Predictions": int(errs.size),
                "MAE (s)": float(np.mean(np.abs(errs))),
                "RMSE (s)": float(np.sqrt(np.mean(errs**2))),
                "P90 |Err| (s)": float(np.percentile(np.abs(errs), 90)),
                "Median |Err| (s)": float(np.median(np.abs(errs))),
                "Flagged Trips": int(flagged),
                "Trips Fit": int(n_trip_fit),
                "Flagged %": float(100.0 * flagged / n_trip_fit)
                if n_trip_fit
                else np.nan,
                "Failures": int(failures),
            }
        )

    return pl.from_dicts(summary_rows), pl.from_dicts(detail_rows)

In [ ]:
BACKTRACK_TOL_M = 0.5


def _collapse_backtracking_knots(
    arrivals: list,
    s_m: list[float],
    stop_ids: list,
    backtrack_tol_m: float = BACKTRACK_TOL_M,
) -> tuple[list, list[float], list, int]:
    """Drop knots that do not advance by more than the backtrack tolerance."""
    if not arrivals:
        return arrivals, s_m, stop_ids, 0

    keep_idx = [0]
    collapsed = 0
    for i in range(1, len(s_m)):
        if s_m[i] <= s_m[keep_idx[-1]] + backtrack_tol_m:
            collapsed += 1
            continue
        keep_idx.append(i)

    return (
        [arrivals[i] for i in keep_idx],
        [s_m[i] for i in keep_idx],
        [stop_ids[i] for i in keep_idx],
        collapsed,
    )


def build_trip_knots_from_rt_geo(
    df_rt_geo_input: pl.DataFrame,
    time_tol_s: float,
    dist_tol_m: float,
) -> list[dict]:
    """Build per-trip monotone knots (time, distance) from df_rt_geo under given tolerances."""
    grouped = (
        df_rt_geo_input.sort(["trip_id", "arrival"])
        .group_by(["trip_id", "route_id", "direction"])
        .agg(
            pl.col("arrival"),
            pl.col("cum_distance_m"),
            pl.col("stop_id"),
            pl.col("route_length_m").first().alias("route_length_m"),
        )
    )

    trips_out: list[dict] = []
    total_collapsed = 0
    trips_with_collapsed = 0
    for row in grouped.iter_rows(named=True):
        arrivals = list(row["arrival"])
        s_m = list(row["cum_distance_m"])
        stop_ids = list(row["stop_id"])
        L = float(row["route_length_m"])

        if not arrivals:
            continue

        # Sort by arrival and dedupe ties within TIME_TOL_S, keeping the latest point in each cluster.
        order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
        arrivals = [arrivals[i] for i in order]
        s_m = [float(s_m[i]) for i in order]
        stop_ids = [stop_ids[i] for i in order]

        a_sec = [(a - arrivals[0]).total_seconds() for a in arrivals]
        keep_idx = [len(a_sec) - 1]
        for i in range(len(a_sec) - 2, -1, -1):
            if a_sec[keep_idx[-1]] - a_sec[i] > time_tol_s:
                keep_idx.append(i)
        keep_idx = sorted(keep_idx)

        arrivals = [arrivals[i] for i in keep_idx]
        s_m = [s_m[i] for i in keep_idx]
        stop_ids = [stop_ids[i] for i in keep_idx]

        n = len(arrivals)
        if n < 4:
            continue

        if not all(arrivals[i] > arrivals[i - 1] for i in range(1, n)):
            continue

        kind = _classify_monotone(s_m, dist_tol_m)
        if kind == "none":
            continue
        if kind == "dec":
            s_m = [L - x for x in s_m]

        arrivals, s_m, stop_ids, collapsed = _collapse_backtracking_knots(
            arrivals,
            s_m,
            stop_ids,
            backtrack_tol_m=BACKTRACK_TOL_M,
        )
        if collapsed:
            total_collapsed += collapsed
            trips_with_collapsed += 1
            print(
                f"[knot-collapse] trip_id={row['trip_id']} route_id={row['route_id']} "
                f"direction={row['direction']} collapsed {collapsed} knot(s) "
                f"(tol={BACKTRACK_TOL_M:.2f}m)"
            )

        n = len(arrivals)
        if n < 4:
            continue

        t_base = np.asarray(
            [(a - arrivals[0]).total_seconds() for a in arrivals],
            dtype=float,
        )

        trips_out.append(
            {
                "trip_id": row["trip_id"],
                "route_id": row["route_id"],
                "direction": row["direction"],
                "route_length_m": L,
                "stop_ids": stop_ids,
                "t_base_sec": t_base,
                "s_m": np.asarray(s_m, dtype=float),
                "n_stops": n,
            }
        )

    if total_collapsed:
        print(
            f"[knot-collapse] collapsed {total_collapsed} knot(s) across {trips_with_collapsed} trip(s)"
        )
    else:
        print(
            f"[knot-collapse] no backwards knots collapsed (tol={BACKTRACK_TOL_M:.2f}m)"
        )

    return trips_out


In [ ]:
# 1) Dwell sensitivity sweep.
base_trip_knots = build_trip_knots_from_rt_geo(
    df_rt_geo_input=df_rt_geo,
    time_tol_s=TIME_TOL_S,
    dist_tol_m=DIST_TOL_M,
)

sweep_trip_knots = sample_trip_knots_by_route(
    base_trip_knots,
    max_trips_per_route=SWEEP_MAX_TRIPS_PER_ROUTE,
    seed=RNG_SEED,
)

DWELL_SWEEP_S = [10, 20, 30, 40, 50]
dwell_metric_rows: list[dict] = []

for dwell_s in DWELL_SWEEP_S:
    trips_variant = apply_dwell_to_trip_knots(sweep_trip_knots, dwell_s=dwell_s)
    df_summ, _ = evaluate_methods_on_trip_knots(
        trips_variant,
        methods=METHODS_TO_COMPARE,
        v_max=SANITY_V_MAX,
        a_max=SANITY_A_MAX,
    )
    df_summ = df_summ.with_columns(pl.lit(float(dwell_s)).alias("dwell_s"))
    dwell_metric_rows.extend(df_summ.to_dicts())

df_dwell_sweep = pl.from_dicts(dwell_metric_rows).with_columns(
    pl.col("dwell_s").cast(pl.Int64)
)

print("=== Dwell sensitivity sweep (sampled trips) ===")
print(f"Sampled trips: {len(sweep_trip_knots)}")
display(
    df_dwell_sweep.select(
        ["dwell_s", "Method", "Predictions", "MAE (s)", "P90 |Err| (s)", "Flagged %"]
    )
    .sort(["Method", "dwell_s"])
    .with_columns(pl.all().exclude(["Method", "Predictions", "dwell_s"]).round(3))
)

# Plot MAE vs dwell for each method.
fig, ax = plt.subplots(figsize=(11, 5))
for method in METHODS_TO_COMPARE:
    df_m = df_dwell_sweep.filter(pl.col("Method") == method).sort("dwell_s")
    ax.plot(
        df_m.get_column("dwell_s").to_numpy(),
        df_m.get_column("MAE (s)").to_numpy(),
        marker="o",
        linewidth=1.8,
        label=method,
    )

ax.set_xlabel("Artificial Dwell (seconds)")
ax.set_ylabel("MAE (seconds)")
ax.set_title("Dwell Sensitivity: MAE vs Synthetic Dwell")
ax.grid(alpha=0.25)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

# Robustness summary for dwell sweep.
df_dwell_robustness = (
    df_dwell_sweep.group_by("Method")
    .agg(
        pl.col("MAE (s)").mean().alias("Mean MAE (s)"),
        pl.col("MAE (s)").std().alias("Std MAE (s)"),
        pl.col("MAE (s)").min().alias("Best MAE (s)"),
        pl.col("MAE (s)").max().alias("Worst MAE (s)"),
    )
    .with_columns(
        (pl.col("Std MAE (s)") / pl.col("Mean MAE (s)")).alias("Dwell MAE CV")
    )
    .sort("Dwell MAE CV")
    .with_columns(pl.all().exclude(["Method"]).round(4))
)

print("=== Dwell robustness summary ===")
display(df_dwell_robustness)

In [ ]:
ARTIFICIAL_DWELL_S = 30.0

# 2) Dedupe tolerance sensitivity sweep.
TOLERANCE_SWEEP = [
    {"label": "baseline", "time_tol_s": 0.50, "dist_tol_m": 1.00},
    {"label": "tight_time", "time_tol_s": 0.25, "dist_tol_m": 1.00},
    {"label": "loose_time", "time_tol_s": 1.00, "dist_tol_m": 1.00},
    {"label": "tight_dist", "time_tol_s": 0.50, "dist_tol_m": 0.50},
    {"label": "loose_dist", "time_tol_s": 0.50, "dist_tol_m": 2.00},
]

tol_metric_rows: list[dict] = []
for cfg in TOLERANCE_SWEEP:
    knots_cfg = build_trip_knots_from_rt_geo(
        df_rt_geo_input=df_rt_geo,
        time_tol_s=float(cfg["time_tol_s"]),
        dist_tol_m=float(cfg["dist_tol_m"]),
    )
    knots_cfg = sample_trip_knots_by_route(
        knots_cfg,
        max_trips_per_route=SWEEP_MAX_TRIPS_PER_ROUTE,
        seed=RNG_SEED,
    )
    knots_cfg = apply_dwell_to_trip_knots(knots_cfg, dwell_s=ARTIFICIAL_DWELL_S)

    df_summ, _ = evaluate_methods_on_trip_knots(
        knots_cfg,
        methods=METHODS_TO_COMPARE,
        v_max=SANITY_V_MAX,
        a_max=SANITY_A_MAX,
    )

    df_summ = df_summ.with_columns(
        pl.lit(cfg["label"]).alias("tol_label"),
        pl.lit(float(cfg["time_tol_s"])).alias("time_tol_s"),
        pl.lit(float(cfg["dist_tol_m"])).alias("dist_tol_m"),
        pl.lit(len(knots_cfg)).alias("Trips Sampled"),
    )
    tol_metric_rows.extend(df_summ.to_dicts())

df_tol_sweep = pl.from_dicts(tol_metric_rows)

print("=== Tolerance sensitivity sweep (sampled trips) ===")
display(
    df_tol_sweep.select(
        [
            "tol_label",
            "time_tol_s",
            "dist_tol_m",
            "Method",
            "Trips Sampled",
            "MAE (s)",
            "P90 |Err| (s)",
            "Flagged %",
        ]
    )
    .sort(["Method", "tol_label"])
    .with_columns(pl.all().exclude(["tol_label", "Method", "Trips Sampled"]).round(3))
)

# Plot MAE drift relative to baseline tolerance for each method.
baseline = df_tol_sweep.filter(pl.col("tol_label") == "baseline").select(
    ["Method", pl.col("MAE (s)").alias("baseline_mae")]
)

df_tol_delta = df_tol_sweep.join(baseline, on="Method", how="left").with_columns(
    ((pl.col("MAE (s)") - pl.col("baseline_mae")) / pl.col("baseline_mae") * 100).alias(
        "MAE Delta %"
    )
)

fig, ax = plt.subplots(figsize=(11, 5))
for method in METHODS_TO_COMPARE:
    df_m = df_tol_delta.filter(pl.col("Method") == method)
    x = np.arange(df_m.height)
    ax.plot(
        x,
        df_m.get_column("MAE Delta %").to_numpy(),
        marker="o",
        linewidth=1.8,
        label=method,
    )

ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
ax.set_xticks(np.arange(len(TOLERANCE_SWEEP)))
ax.set_xticklabels([c["label"] for c in TOLERANCE_SWEEP], rotation=20, ha="right")
ax.set_ylabel("MAE Change vs Baseline (%)")
ax.set_title("Tolerance Sensitivity: Relative MAE Drift")
ax.grid(alpha=0.25)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

# Robustness summary for tolerance sweep.
df_tol_robustness = (
    df_tol_delta.group_by("Method")
    .agg(
        pl.col("MAE Delta %").abs().mean().alias("Mean |MAE Delta| %"),
        pl.col("MAE Delta %").abs().max().alias("Worst |MAE Delta| %"),
    )
    .sort("Mean |MAE Delta| %")
    .with_columns(pl.all().exclude(["Method"]).round(4))
)

print("=== Tolerance robustness summary ===")
display(df_tol_robustness)

In [ ]:
# 3) Final weighted model-selection rubric.

# Baseline components (full dataset summaries already computed above).
df_rank_base = (
    df_cv_method.select(["Method", "MAE (s)", "P90 |Err| (s)"])
    .join(df_physics.select(["Method", "% Flagged"]), on="Method", how="left")
    .join(
        df_route_fairness.select(["Method", "Route IQR (s)"]), on="Method", how="left"
    )
)

# Robustness components from the two sweeps.
df_rank_rob = (
    df_dwell_robustness.select(["Method", "Dwell MAE CV"])
    .join(
        df_tol_robustness.select(["Method", "Mean |MAE Delta| %"]),
        on="Method",
        how="left",
    )
    .with_columns(
        (pl.col("Dwell MAE CV") + pl.col("Mean |MAE Delta| %") / 100.0).alias(
            "Robustness Penalty"
        )
    )
)

# Merge all ranking ingredients.
df_rank_raw = df_rank_base.join(df_rank_rob, on="Method", how="left")


def minmax_norm(series: np.ndarray) -> np.ndarray:
    smin = np.nanmin(series)
    smax = np.nanmax(series)
    if not np.isfinite(smin) or not np.isfinite(smax) or smax <= smin:
        return np.zeros_like(series, dtype=float)
    return (series - smin) / (smax - smin)


rank_pd = df_rank_raw.to_pandas()

# Lower-is-better metrics => normalized penalty in [0, 1].
rank_pd["n_mae"] = minmax_norm(rank_pd["MAE (s)"].to_numpy(dtype=float))
rank_pd["n_p90"] = minmax_norm(rank_pd["P90 |Err| (s)"].to_numpy(dtype=float))
rank_pd["n_flag"] = minmax_norm(rank_pd["% Flagged"].to_numpy(dtype=float))
rank_pd["n_fair"] = minmax_norm(rank_pd["Route IQR (s)"].to_numpy(dtype=float))
rank_pd["n_rob"] = minmax_norm(rank_pd["Robustness Penalty"].to_numpy(dtype=float))

# Weights for report-facing selection rubric.
W_MAE = 0.30
W_P90 = 0.15
W_PHYS = 0.25
W_FAIR = 0.15
W_ROB = 0.15

rank_pd["Penalty Score"] = (
    W_MAE * rank_pd["n_mae"]
    + W_P90 * rank_pd["n_p90"]
    + W_PHYS * rank_pd["n_flag"]
    + W_FAIR * rank_pd["n_fair"]
    + W_ROB * rank_pd["n_rob"]
)
rank_pd["Composite Score (0-100, higher better)"] = 100.0 * (
    1.0 - rank_pd["Penalty Score"]
)

rank_pd = rank_pd.sort_values("Composite Score (0-100, higher better)", ascending=False)
rank_pd["Rank"] = np.arange(1, len(rank_pd) + 1)

df_model_ranking = pl.from_pandas(
    rank_pd[
        [
            "Rank",
            "Method",
            "Composite Score (0-100, higher better)",
            "MAE (s)",
            "P90 |Err| (s)",
            "% Flagged",
            "Route IQR (s)",
            "Dwell MAE CV",
            "Mean |MAE Delta| %",
            "Robustness Penalty",
        ]
    ]
).with_columns(pl.all().exclude(["Rank", "Method"]).cast(pl.Float64).round(4))

print("=== Final model ranking (weighted rubric) ===")
print(
    "Weights: "
    f"MAE={W_MAE:.2f}, P90={W_P90:.2f}, Physics={W_PHYS:.2f}, "
    f"Fairness={W_FAIR:.2f}, Robustness={W_ROB:.2f}"
)
display(df_model_ranking)

# Ranking bar chart.
fig, ax = plt.subplots(figsize=(10, 5))
methods_ranked = df_model_ranking.get_column("Method").to_list()
scores_ranked = df_model_ranking.get_column(
    "Composite Score (0-100, higher better)"
).to_numpy()
ax.bar(methods_ranked, scores_ranked)
ax.set_ylabel("Composite Score (0-100)")
ax.set_title("Final Interpolation Method Ranking")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## Auto-Generated Report Summary

This cell composes a report-ready narrative directly from the computed evaluation tables so your written conclusions always stay synced with notebook results.


In [ ]:
from IPython.display import Markdown

rank_rows = df_model_ranking.sort("Rank").to_dicts()
top1, top2, top3 = rank_rows[0], rank_rows[1], rank_rows[2]

best_method = top1["Method"]
score_gap_12 = float(
    top1["Composite Score (0-100, higher better)"]
    - top2["Composite Score (0-100, higher better)"]
)

best_holdout = (
    df_holdout_gap.filter(pl.col("Method") == best_method)
    .select(["Train MAE (s)", "Test MAE (s)", "MAE Gap (s)"])
    .to_dicts()[0]
)

best_phys = (
    df_physics.filter(pl.col("Method") == best_method)
    .select(["% Flagged", "Median V_max (m/s)", "95th Pct A_max (m/s^2)"])
    .to_dicts()[0]
)

best_route_perf = df_cv_route_method.filter(pl.col("Method") == best_method).select(
    ["route_id", "Median |Err| (s)"]
)
hardest_route = best_route_perf.sort("Median |Err| (s)", descending=True).to_dicts()[0]
easier_route = best_route_perf.sort("Median |Err| (s)").to_dicts()[0]

best_dwell = df_dwell_sweep.filter(pl.col("Method") == best_method).sort("dwell_s")
dwell_mae_min = float(best_dwell.get_column("MAE (s)").min())
dwell_mae_max = float(best_dwell.get_column("MAE (s)").max())
dwell_flag_min = float(best_dwell.get_column("Flagged %").min())
dwell_flag_max = float(best_dwell.get_column("Flagged %").max())

best_tol = (
    df_tol_robustness.filter(pl.col("Method") == best_method)
    .select(["Mean |MAE Delta| %", "Worst |MAE Delta| %"])
    .to_dicts()[0]
)

route_dom = df_route_coverage.sort("Prediction Share (%)", descending=True).to_dicts()[
    0
]

report_md = f"""
### Final Results Summary (Auto-generated)

Across the evaluated interpolation methods, **{best_method}** ranked first under the weighted rubric (composite score **{top1["Composite Score (0-100, higher better)"]:.2f}/100**), followed by **{top2["Method"]}** and **{top3["Method"]}**. The margin between the top two methods was **{score_gap_12:.2f}** points, indicating {"a clear" if score_gap_12 >= 2 else "a close"} lead for {best_method} under the current weighting.

For out-of-split consistency, {best_method} had train MAE **{best_holdout["Train MAE (s)"]:.2f}s** and test MAE **{best_holdout["Test MAE (s)"]:.2f}s**, with a generalization gap of **{best_holdout["MAE Gap (s)"]:.2f}s**. This suggests that performance remains broadly stable when moving from the 80% split to held-out trips.

Physics plausibility for {best_method} showed **{best_phys["% Flagged"]:.2f}%** flagged trips, with median peak speed **{best_phys["Median V_max (m/s)"]:.2f} m/s** and 95th-percentile peak acceleration **{best_phys["95th Pct A_max (m/s^2)"]:.3f} m/s^2**. Route-level behavior was not uniform: hardest route for {best_method} was **{hardest_route["route_id"]}** (median absolute error **{hardest_route["Median |Err| (s)"]:.2f}s**) while easiest was **{easier_route["route_id"]}** (**{easier_route["Median |Err| (s)"]:.2f}s**).

Robustness checks showed that for {best_method}, MAE across dwell settings ranged from **{dwell_mae_min:.2f}s** to **{dwell_mae_max:.2f}s**, while flagged-rate ranged from **{dwell_flag_min:.2f}%** to **{dwell_flag_max:.2f}%**. Under dedupe-tolerance perturbations, mean absolute MAE drift was **{best_tol["Mean |MAE Delta| %"]:.2f}%** (worst case **{best_tol["Worst |MAE Delta| %"]:.2f}%**), indicating moderate sensitivity but no severe instability.

Dataset coverage remained somewhat imbalanced by route: **{route_dom["route_id"]}** contributed **{route_dom["Prediction Share (%)"]:.2f}%** of CV predictions. This should be acknowledged as a limitation when generalizing conclusions to all routes equally.

**Recommendation:** Based on the current weighting and robustness diagnostics, use **{best_method}** as the primary interpolation method for continuous tracking, with **{top2["Method"]}** as a strong fallback candidate.
""".strip()

report_plain = (
    f"Top methods: {top1['Method']} > {top2['Method']} > {top3['Method']}. "
    f"Top score {top1['Composite Score (0-100, higher better)']:.2f}/100; gap to second {score_gap_12:.2f}. "
    f"{best_method} holdout MAE: train {best_holdout['Train MAE (s)']:.2f}s, test {best_holdout['Test MAE (s)']:.2f}s, gap {best_holdout['MAE Gap (s)']:.2f}s. "
    f"Physics flagged {best_phys['% Flagged']:.2f}%. "
    f"Hardest route for {best_method}: {hardest_route['route_id']} ({hardest_route['Median |Err| (s)']:.2f}s), "
    f"easiest: {easier_route['route_id']} ({easier_route['Median |Err| (s)']:.2f}s). "
    f"Dwell MAE range {dwell_mae_min:.2f}-{dwell_mae_max:.2f}s; tolerance drift mean {best_tol['Mean |MAE Delta| %']:.2f}% (worst {best_tol['Worst |MAE Delta| %']:.2f}%). "
    f"Largest route share: {route_dom['route_id']} at {route_dom['Prediction Share (%)']:.2f}% of predictions."
)

display(Markdown(report_md))
print("\n--- Plaintext version ---\n")
print(report_plain)

## Visual Quality Proxy (Deck.gl-Oriented, No Deck.gl Dependency)

This section estimates which interpolation method would _look_ best in a trip animation by scoring:

- Smoothness: acceleration and jerk spikes
- Temporal stability: frame-to-frame speed jitter
- Visual plausibility: backtracking frequency
- Track adherence: how closely each method passes through observed knots

These are proxy metrics for map animation quality and do not require Deck.gl runtime integration.


In [ ]:
VISUAL_DT_S = 1.0
VISUAL_EPS_BACKTRACK_M = 0.5

# Use the same sampled set used in robustness sweeps so runtime stays manageable.
trip_knots_visual = apply_dwell_to_trip_knots(
    sweep_trip_knots, dwell_s=ARTIFICIAL_DWELL_S
)


def minmax_penalty(x: np.ndarray) -> np.ndarray:
    lo = np.nanmin(x)
    hi = np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)


visual_rows: list[dict] = []
for method in METHODS_TO_COMPARE:
    for trip in trip_knots_visual:
        t_sec = np.asarray(trip["t_sec"], dtype=float)
        s_m = np.asarray(trip["s_m"], dtype=float)
        if t_sec.size < 5:
            continue

        try:
            f = get_interpolator(method, t_sec, s_m)

            ts = np.arange(t_sec[0], t_sec[-1] + VISUAL_DT_S * 0.5, VISUAL_DT_S)
            if ts.size < 5:
                continue

            s_hat = np.asarray(f(ts), dtype=float)
            if np.any(~np.isfinite(s_hat)):
                continue

            v = np.gradient(s_hat, VISUAL_DT_S)
            a = np.gradient(v, VISUAL_DT_S)
            j = np.gradient(a, VISUAL_DT_S)

            dv = np.diff(v)
            ds = np.diff(s_hat)

            # How much the fitted curve misses observed knots in distance-space.
            s_at_knots = np.asarray([float(f(float(ti))) for ti in t_sec], dtype=float)
            knot_mae_m = float(np.mean(np.abs(s_at_knots - s_m)))

            # Visual smoothness proxies designed to catch sparse but harsh jumps.
            speed_cv = float(np.std(v) / (np.mean(np.abs(v)) + 1e-6))
            speed_tv = float(np.mean(np.abs(dv))) if dv.size else 0.0
            speed_delta_p99 = float(np.percentile(np.abs(dv), 99)) if dv.size else 0.0

            accel_rms = float(np.sqrt(np.mean(a**2)))
            accel_p99 = float(np.percentile(np.abs(a), 99))
            jerk_rms = float(np.sqrt(np.mean(j**2)))
            jerk_p99 = float(np.percentile(np.abs(j), 99))

            backtrack_pct = (
                float(100.0 * np.mean(ds < -VISUAL_EPS_BACKTRACK_M)) if ds.size else 0.0
            )

            visual_rows.append(
                {
                    "Method": method,
                    "trip_id": trip["trip_id"],
                    "route_id": trip["route_id"],
                    "knot_mae_m": knot_mae_m,
                    "speed_cv": speed_cv,
                    "speed_tv": speed_tv,
                    "speed_delta_p99": speed_delta_p99,
                    "accel_rms": accel_rms,
                    "accel_p99": accel_p99,
                    "jerk_rms": jerk_rms,
                    "jerk_p99": jerk_p99,
                    "backtrack_pct": backtrack_pct,
                }
            )
        except Exception:
            continue

if not visual_rows:
    raise RuntimeError("No visual-proxy rows produced; check input trip knots.")

df_visual_detail = pl.from_dicts(visual_rows)

df_visual_method = (
    df_visual_detail.group_by("Method")
    .agg(
        pl.len().alias("Trips Evaluated"),
        pl.col("knot_mae_m").median().alias("Knot MAE median (m)"),
        pl.col("speed_cv").median().alias("Speed CV median"),
        pl.col("speed_tv").median().alias("Speed TV median"),
        pl.col("speed_delta_p99").median().alias("Speed Delta P99 median"),
        pl.col("accel_rms").median().alias("Accel RMS median"),
        pl.col("accel_p99").median().alias("Accel P99 median"),
        pl.col("jerk_rms").median().alias("Jerk RMS median"),
        pl.col("jerk_p99").median().alias("Jerk P99 median"),
        pl.col("backtrack_pct").mean().alias("Backtrack mean (%)"),
    )
    .sort("Method")
)

# Build an animation-quality score where lower penalties are better.
vp = df_visual_method.to_pandas().copy()

vp["p_knot"] = minmax_penalty(vp["Knot MAE median (m)"].to_numpy(dtype=float))
vp["p_speed_cv"] = minmax_penalty(vp["Speed CV median"].to_numpy(dtype=float))
vp["p_speed_tv"] = minmax_penalty(vp["Speed TV median"].to_numpy(dtype=float))
vp["p_speed_delta"] = minmax_penalty(vp["Speed Delta P99 median"].to_numpy(dtype=float))
vp["p_accel_rms"] = minmax_penalty(vp["Accel RMS median"].to_numpy(dtype=float))
vp["p_accel_p99"] = minmax_penalty(vp["Accel P99 median"].to_numpy(dtype=float))
vp["p_jerk_rms"] = minmax_penalty(vp["Jerk RMS median"].to_numpy(dtype=float))
vp["p_jerk_p99"] = minmax_penalty(vp["Jerk P99 median"].to_numpy(dtype=float))
vp["p_backtrack"] = minmax_penalty(vp["Backtrack mean (%)"].to_numpy(dtype=float))

# Heavier weight on continuity and suppression of high-frequency jitter.
WV_KNOT = 0.05
WV_SPEED_CV = 0.10
WV_SPEED_TV = 0.15
WV_SPEED_DELTA = 0.15
WV_ACCEL_RMS = 0.10
WV_ACCEL_P99 = 0.10
WV_JERK_RMS = 0.15
WV_JERK_P99 = 0.15
WV_BACKTRACK = 0.05

vp["Visual Penalty"] = (
    WV_KNOT * vp["p_knot"]
    + WV_SPEED_CV * vp["p_speed_cv"]
    + WV_SPEED_TV * vp["p_speed_tv"]
    + WV_SPEED_DELTA * vp["p_speed_delta"]
    + WV_ACCEL_RMS * vp["p_accel_rms"]
    + WV_ACCEL_P99 * vp["p_accel_p99"]
    + WV_JERK_RMS * vp["p_jerk_rms"]
    + WV_JERK_P99 * vp["p_jerk_p99"]
    + WV_BACKTRACK * vp["p_backtrack"]
)
vp["Visual Score (0-100, higher better)"] = 100.0 * (1.0 - vp["Visual Penalty"])
vp = vp.sort_values("Visual Score (0-100, higher better)", ascending=False)
vp["Visual Rank"] = np.arange(1, len(vp) + 1)

df_visual_ranking = pl.from_pandas(
    vp[
        [
            "Visual Rank",
            "Method",
            "Visual Score (0-100, higher better)",
            "Knot MAE median (m)",
            "Speed CV median",
            "Speed TV median",
            "Speed Delta P99 median",
            "Accel RMS median",
            "Accel P99 median",
            "Jerk RMS median",
            "Jerk P99 median",
            "Backtrack mean (%)",
        ]
    ]
).with_columns(pl.all().exclude(["Visual Rank", "Method"]).cast(pl.Float64).round(4))

print("=== Visual-proxy method ranking (animation-oriented) ===")
print(f"Trips evaluated: {df_visual_detail['trip_id'].n_unique():,}")
print(
    "Weights: "
    f"knot={WV_KNOT:.2f}, speed_cv={WV_SPEED_CV:.2f}, speed_tv={WV_SPEED_TV:.2f}, "
    f"speed_delta_p99={WV_SPEED_DELTA:.2f}, accel_rms={WV_ACCEL_RMS:.2f}, "
    f"accel_p99={WV_ACCEL_P99:.2f}, jerk_rms={WV_JERK_RMS:.2f}, "
    f"jerk_p99={WV_JERK_P99:.2f}, backtrack={WV_BACKTRACK:.2f}"
)
display(df_visual_ranking)

# Plot: visual score by method.
fig, ax = plt.subplots(figsize=(10, 5))
m = df_visual_ranking.get_column("Method").to_list()
s = df_visual_ranking.get_column("Visual Score (0-100, higher better)").to_numpy()
ax.bar(m, s)
ax.set_ylabel("Visual Score (0-100)")
ax.set_ylim(0, 100)
ax.set_title("Animation-Oriented Visual Quality Ranking")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Debug why trips fail the extreme segment velocity filter.
# This mirrors the filtering logic and records the worst segment per dropped trip.

velocity_debug_rows: list[dict] = []
reason_counts = {
    "too_short": 0,
    "nonmono_time": 0,
    "nonmono_dist": 0,
    "extreme_velocity": 0,
    "valid": 0,
}

for row in df_trips_grouped.iter_rows(named=True):
    arrivals = list(row["arrival"])
    s_m = list(row["cum_distance_m"])
    stop_ids = list(row["stop_id"])
    route_id = row["route_id"]
    trip_id = row["trip_id"]
    L = float(row["route_length_m"])

    # Keep same sort + time dedupe behavior as production cell.
    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    arrivals = [arrivals[i] for i in order]
    s_m = [s_m[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    a_sec = [(a - arrivals[0]).total_seconds() for a in arrivals]
    keep_idx = [0]
    for i in range(1, len(a_sec)):
        if a_sec[i] - a_sec[keep_idx[-1]] > TIME_TOL_S:
            keep_idx.append(i)

    arrivals = [arrivals[i] for i in keep_idx]
    s_m = [float(s_m[i]) for i in keep_idx]
    stop_ids = [stop_ids[i] for i in keep_idx]

    if len(arrivals) < 4:
        reason_counts["too_short"] += 1
        continue

    if not all(arrivals[i] > arrivals[i - 1] for i in range(1, len(arrivals))):
        reason_counts["nonmono_time"] += 1
        continue

    kind = _classify_monotone(s_m, DIST_TOL_M)
    if kind == "none":
        reason_counts["nonmono_dist"] += 1
        continue

    if kind == "dec":
        s_m = [L - x for x in s_m]

    keep = [0]
    for i in range(1, len(s_m)):
        if s_m[i] - s_m[keep[-1]] > DIST_TOL_M:
            keep.append(i)

    arrivals = [arrivals[i] for i in keep]
    s_m = [s_m[i] for i in keep]
    stop_ids = [stop_ids[i] for i in keep]

    if len(arrivals) < 4:
        reason_counts["too_short"] += 1
        continue

    a_sec_final = np.asarray(
        [(a - arrivals[0]).total_seconds() for a in arrivals], dtype=float
    )
    dt = np.diff(a_sec_final)
    ds = np.diff(np.asarray(s_m, dtype=float))

    if dt.size == 0:
        reason_counts["too_short"] += 1
        continue

    seg_v = ds / np.where(dt > 0, dt, 1e-6)
    abs_v = np.abs(seg_v)
    vmax = float(np.max(abs_v))
    imax = int(np.argmax(abs_v))

    if vmax > SANITY_V_MAX:
        reason_counts["extreme_velocity"] += 1
        velocity_debug_rows.append(
            {
                "trip_id": trip_id,
                "route_id": route_id,
                "max_seg_v_mps": vmax,
                "max_seg_v_mph": vmax * 2.236936,
                "dt_s": float(dt[imax]),
                "ds_m": float(ds[imax]),
                "abs_ds_m": float(abs(ds[imax])),
                "from_stop": stop_ids[imax],
                "to_stop": stop_ids[imax + 1],
                "n_points_after_dedupe": len(arrivals),
            }
        )
    else:
        reason_counts["valid"] += 1


df_velocity_debug = pl.from_dicts(velocity_debug_rows).sort(
    "max_seg_v_mps", descending=True
)

print("=== Drop reason counts (reconstructed) ===")
print(reason_counts)

if df_velocity_debug.height > 0:
    print("\n=== Extreme-velocity trip summary ===")
    display(
        df_velocity_debug.select(
            [
                pl.len().alias("n_extreme_trips"),
                pl.col("route_id").n_unique().alias("routes_with_extremes"),
                pl.col("max_seg_v_mps").median().alias("median_max_v_mps"),
                pl.col("max_seg_v_mps").quantile(0.90).alias("p90_max_v_mps"),
                pl.col("max_seg_v_mps").quantile(0.99).alias("p99_max_v_mps"),
                pl.col("max_seg_v_mps").max().alias("max_max_v_mps"),
                pl.col("dt_s").median().alias("median_dt_s_at_violation"),
                pl.col("dt_s").quantile(0.10).alias("p10_dt_s_at_violation"),
                pl.col("abs_ds_m").median().alias("median_abs_ds_m_at_violation"),
                pl.col("abs_ds_m").quantile(0.90).alias("p90_abs_ds_m_at_violation"),
            ]
        )
    )

    print("\n=== Extreme trips by route ===")
    display(
        df_velocity_debug.group_by("route_id")
        .agg(
            pl.len().alias("n_extreme"),
            pl.col("max_seg_v_mps").median().alias("median_max_v_mps"),
            pl.col("dt_s").median().alias("median_dt_s"),
            pl.col("abs_ds_m").median().alias("median_abs_ds_m"),
        )
        .sort("n_extreme", descending=True)
    )

    print("\n=== dt bins at offending segment ===")
    display(
        df_velocity_debug.with_columns(
            pl.when(pl.col("dt_s") < 1)
            .then(pl.lit("<1s"))
            .when(pl.col("dt_s") < 2)
            .then(pl.lit("1-2s"))
            .when(pl.col("dt_s") < 5)
            .then(pl.lit("2-5s"))
            .when(pl.col("dt_s") < 10)
            .then(pl.lit("5-10s"))
            .when(pl.col("dt_s") < 20)
            .then(pl.lit("10-20s"))
            .otherwise(pl.lit(">=20s"))
            .alias("dt_bin")
        )
        .group_by("dt_bin")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
    )

    print("\n=== Top 20 worst offending segments ===")
    display(df_velocity_debug.head(20))
else:
    print("No extreme velocity trips found in reconstruction.")